<a href="https://colab.research.google.com/github/jhenningsen/Equity_Analysis/blob/main/MomentumRotation/MomentumRotation_v1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### SMA-Based Daily Recommendation Script

This script is structured to fulfill your requirements. It focuses on generating a daily 'buy' recommendation for a single ETF based on a simple SMA-crossing strategy. It doesn't execute trades but prints the recommended ticker, simulating the decision-making process.

**Key Features:**

*   **Rule-Based Trading**: The core logic is encapsulated in the `generate_recommendation` method, scheduled to run daily before market close.
*   **Warm-up Periods**: Ensures that all SMAs have enough historical data before recommendations begin.
*   **Universe Definition**: Starts with a predefined list of 6 commonly traded ETFs (SPY, QQQ, DIA, IWM, GLD, TLT).
*   **Indicator-Driven**: Utilizes 3, 5, 10, and 20-day Simple Moving Averages (SMAs).
*   **Concentrated Positions**: Identifies and recommends only one best-performing ETF each day based on the SMA criteria.

**Recommendation Logic:**

For each ETF, a 'score' is calculated. This score increases if:
1.  The current price is above its 3-day SMA.
2.  The 3-day SMA is above the 5-day SMA.
3.  The 5-day SMA is above the 10-day SMA.
4.  The 10-day SMA is above the 20-day SMA.

The ETF with the highest score (indicating a strong upward trend across multiple timeframes) is selected as the daily recommendation. If multiple ETFs have the same highest score, the first one encountered will be chosen.


In [23]:
# Install yfinance if you haven't already
!pip install yfinance

import yfinance as yf
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

class SMABasedRecommendationColab:

    def __init__(self, tickers=None, sma_periods=None, start_date=None, end_date=None):
        # === CONFIGURATION ===
        self.universe_tickers = tickers if tickers is not None else ["SPY", "QQQ", "DIA", "IWM", "GLD", "TLT"]
        self.sma_periods = sma_periods if sma_periods is not None else [3, 5, 10, 20]
        self.start_date = pd.to_datetime(start_date) if start_date else pd.to_datetime('2020-01-01')
        self.end_date = pd.to_datetime(end_date) if end_date else pd.to_datetime('2026-01-01')

        # === DATA STORAGE ===
        self.historical_data = pd.DataFrame() # To store all downloaded price data
        self.smas_data = {} # To store SMA values for each ticker
        self.recommendations = [] # To store daily recommendations

        # === WARM-UP ===
        # We need enough data to calculate the longest SMA
        self.warmup_days = max(self.sma_periods) + 5 # +5 days for buffer

        print("Initializing data download...")
        self._download_data()
        print("Data download complete. Calculating SMAs...")
        self._calculate_all_smas()
        print("SMAs calculated. Ready for simulation.")


    def _download_data(self):
        # Download data for all tickers
        # We need data starting from before the actual start_date due to warm-up
        data_start = self.start_date - timedelta(days=self.warmup_days * 2) # Get extra data for safety
        # Changed 'Adj Close' to 'Close' due to yfinance auto_adjust=True default
        self.historical_data = yf.download(self.universe_tickers, start=data_start, end=self.end_date, interval='1d')['Close']

        # Ensure we have enough data for the entire period after warm-up
        min_data_points = max(self.sma_periods)
        self.historical_data = self.historical_data.dropna(axis=1, thresh=min_data_points) # Drop tickers with insufficient data
        self.universe_tickers = self.historical_data.columns.tolist() # Update tickers based on available data
        if not self.universe_tickers:
            raise ValueError("No sufficient data downloaded for any ticker in the universe.")

    def _calculate_all_smas(self):
        for ticker in self.universe_tickers:
            self.smas_data[ticker] = pd.DataFrame(index=self.historical_data.index)
            for period in self.sma_periods:
                self.smas_data[ticker][f'SMA_{period}'] = self.historical_data[ticker].rolling(window=period).mean()

    def _generate_daily_recommendation(self, current_date):
        candidate_scores = {}

        for ticker in self.universe_tickers:
            # Get current price and SMA values for the specific date
            current_price_series = self.historical_data.loc[current_date, ticker]
            if pd.isna(current_price_series): # Skip if no data for this date
                continue
            current_price = current_price_series

            sma_values = {}
            smas_ready = True
            for period in self.sma_periods:
                sma_series = self.smas_data[ticker].loc[current_date, f'SMA_{period}']
                if pd.isna(sma_series):
                    smas_ready = False
                    break
                sma_values[period] = sma_series

            if not smas_ready:
                # print(f"DEBUG: SMAs for {ticker} not ready on {current_date.strftime('%Y-%m-%d')}. Skipping.")
                continue

            # Score based on SMA hierarchy: price > SMA3 > SMA5 > SMA10 > SMA20
            score = 0
            if current_price > sma_values[3]:
                score += 1
            if sma_values[3] > sma_values[5]:
                score += 1
            if sma_values[5] > sma_values[10]:
                score += 1
            if sma_values[10] > sma_values[20]:
                score += 1

            candidate_scores[ticker] = score

        if not candidate_scores:
            return None, 0

        # Find the ticker with the highest score
        best_ticker = max(candidate_scores.items(), key=lambda item: item[1])[0]
        highest_score = candidate_scores[best_ticker]

        return best_ticker, highest_score

    def run_simulation(self):
        trade_dates = self.historical_data.index[self.historical_data.index >= self.start_date].normalize().unique()
        warmup_end_date = self.start_date + timedelta(days=self.warmup_days)

        for current_date in trade_dates:
            if current_date < self.start_date: # Ensure we only process from the actual start date
                continue
            if current_date < warmup_end_date: # Simulate warm-up period
                # print(f"Warming up: {current_date.strftime('%Y-%m-%d')}")
                continue

            # QuantConnect-like scheduling: run on trading days
            if current_date.weekday() >= 5: # Skip weekends
                 continue

            # Check if market is open (i.e., data is available for the date)
            if current_date not in self.historical_data.index:
                continue

            recommended_ticker, highest_score = self._generate_daily_recommendation(current_date)

            if recommended_ticker:
                self.recommendations.append({
                    'date': current_date.strftime('%Y-%m-%d'),
                    'recommended_ticker': recommended_ticker,
                    'score': highest_score
                })
                print(f"[{current_date.strftime('%Y-%m-%d')}] Recommendation: {recommended_ticker} (Score: {highest_score})")
            else:
                self.recommendations.append({
                    'date': current_date.strftime('%Y-%m-%d'),
                    'recommended_ticker': 'NONE',
                    'score': 0
                })
                print(f"[{current_date.strftime('%Y-%m-%d')}] No recommendation today.")

        print("\nSimulation Complete.")

# --- How to use it --- #
# 1. Instantiate the class
# rec_engine = SMABasedRecommendationColab(start_date='2023-01-01', end_date='2023-12-31')

# 2. Run the simulation
# rec_engine.run_simulation()

# 3. Access recommendations
# df_recommendations = pd.DataFrame(rec_engine.recommendations)
# display(df_recommendations.head())

### Example Usage:

Run the following cells to see how to use the `SMABasedRecommendationColab` class. This will execute the simulation for a specified date range and display the generated recommendations.

In [24]:
# Instantiate the recommendation engine
# You can customize the tickers, SMA periods, and date range here.
# For example, to run for the full range with default settings:
rec_engine = SMABasedRecommendationColab()

# Or, for a specific short period:
# rec_engine = SMABasedRecommendationColab(start_date='2023-06-01', end_date='2023-06-30')


Initializing data download...


/tmp/ipykernel_32567/3346671377.py:39: FutureWarning: YF.download() has changed argument auto_adjust default to True
  self.historical_data = yf.download(self.universe_tickers, start=data_start, end=self.end_date, interval='1d')['Close']
[*********************100%***********************]  6 of 6 completed


Data download complete. Calculating SMAs...
SMAs calculated. Ready for simulation.


In [25]:
# Run the simulation to generate daily recommendations
rec_engine.run_simulation()


[2020-01-27] Recommendation: GLD (Score: 4)
[2020-01-28] Recommendation: GLD (Score: 3)
[2020-01-29] Recommendation: GLD (Score: 4)
[2020-01-30] Recommendation: TLT (Score: 4)
[2020-01-31] Recommendation: GLD (Score: 4)
[2020-02-03] Recommendation: TLT (Score: 4)
[2020-02-04] Recommendation: QQQ (Score: 4)
[2020-02-05] Recommendation: QQQ (Score: 4)
[2020-02-06] Recommendation: QQQ (Score: 4)
[2020-02-07] Recommendation: QQQ (Score: 3)
[2020-02-10] Recommendation: QQQ (Score: 4)
[2020-02-11] Recommendation: QQQ (Score: 4)
[2020-02-12] Recommendation: DIA (Score: 4)
[2020-02-13] Recommendation: DIA (Score: 4)
[2020-02-14] Recommendation: QQQ (Score: 4)
[2020-02-18] Recommendation: QQQ (Score: 4)
[2020-02-19] Recommendation: GLD (Score: 4)
[2020-02-20] Recommendation: GLD (Score: 4)
[2020-02-21] Recommendation: GLD (Score: 4)
[2020-02-24] Recommendation: GLD (Score: 4)
[2020-02-25] Recommendation: TLT (Score: 4)
[2020-02-26] Recommendation: GLD (Score: 3)
[2020-02-27] Recommendation: TLT

In [26]:
close_prices = []
for index, row in df_recommendations.iterrows():
    date = pd.to_datetime(row['date'])
    ticker = row['recommended_ticker']
    if ticker != 'NONE' and date in rec_engine.historical_data.index:
        try:
            close_price = rec_engine.historical_data.loc[date, ticker]
            close_prices.append(close_price)
        except KeyError:
            close_prices.append(np.nan) # Append NaN if data is missing for some reason
    else:
        close_prices.append(np.nan)

df_recommendations['close_price'] = close_prices
print("\nRecommendations with Close Price:")
display(df_recommendations)


Recommendations with Close Price:


,date,recommended_ticker,score,close_price
0,2020-01-27,GLD,4,148.990005
1,2020-01-28,GLD,3,147.660004
2,2020-01-29,GLD,4,148.460007
3,2020-01-30,TLT,4,120.031151
4,2020-01-31,GLD,4,149.330002
...,...,...,...,...
1589,2026-05-22,DIA,4,NaN
1590,2026-05-26,DIA,4,NaN
1591,2026-05-27,DIA,4,NaN
1592,2026-05-28,DIA,4,NaN


In [27]:
investment_amount = 5000

trades_list = []
current_trade_start_index = -1
current_trade_ticker = None

for i in range(len(df_recommendations)):
    row = df_recommendations.iloc[i]
    current_date = pd.to_datetime(row['date'])
    daily_recommended_ticker = row['recommended_ticker']
    daily_close_price = row['close_price']

    # Case 1: Starting a new trade or continuing an existing one
    if daily_recommended_ticker != 'NONE':
        # If it's a new recommendation or the first entry in a series of recommendations
        if current_trade_ticker is None or daily_recommended_ticker != current_trade_ticker:
            # If there was a previous trade, close it now
            if current_trade_ticker is not None and current_trade_start_index != -1:
                entry_row = df_recommendations.iloc[current_trade_start_index]
                exit_row = df_recommendations.iloc[i - 1] # Previous day is the exit day for the old trade

                entry_ticker = entry_row['recommended_ticker']
                entry_date = pd.to_datetime(entry_row['date'])
                entry_price = entry_row['close_price']

                exit_date = pd.to_datetime(exit_row['date'])
                exit_price = exit_row['close_price']

                if not pd.isna(entry_price) and not pd.isna(exit_price) and entry_price > 0:
                    num_shares = np.floor(investment_amount / entry_price)
                    profit = (exit_price - entry_price) * num_shares
                    trades_list.append({
                        'ticker': entry_ticker,
                        'entry_date': entry_date,
                        'entry_price': entry_price,
                        'exit_date': exit_date,
                        'exit_price': exit_price,
                        'num_shares': num_shares,
                        'profit': profit,
                        'holding_days': (exit_date - entry_date).days + 1
                    })

            # Start the new trade
            current_trade_ticker = daily_recommended_ticker
            current_trade_start_index = i

    # Case 2: Recommendation is 'NONE'
    else: # daily_recommended_ticker == 'NONE'
        # If there was an active trade, close it
        if current_trade_ticker is not None and current_trade_start_index != -1:
            entry_row = df_recommendations.iloc[current_trade_start_index]
            exit_row = df_recommendations.iloc[i - 1] # Previous day is the exit day for the old trade

            entry_ticker = entry_row['recommended_ticker']
            entry_date = pd.to_datetime(entry_row['date'])
            entry_price = entry_row['close_price']

            exit_date = pd.to_datetime(exit_row['date'])
            exit_price = exit_row['close_price']

            if not pd.isna(entry_price) and not pd.isna(exit_price) and entry_price > 0:
                num_shares = np.floor(investment_amount / entry_price)
                profit = (exit_price - entry_price) * num_shares
                trades_list.append({
                    'ticker': entry_ticker,
                    'entry_date': entry_date,
                    'entry_price': entry_price,
                    'exit_date': exit_date,
                    'exit_price': exit_price,
                    'num_shares': num_shares,
                    'profit': profit,
                    'holding_days': (exit_date - entry_date).days + 1
                })
            # Reset trade tracking as there's no recommendation
            current_trade_ticker = None
            current_trade_start_index = -1 # No active trade index

# After the loop, if there's an open trade, close it at the last available day
if current_trade_ticker is not None and current_trade_start_index != -1:
    entry_row = df_recommendations.iloc[current_trade_start_index]
    exit_row = df_recommendations.iloc[len(df_recommendations) - 1] # Last day of the simulation

    entry_ticker = entry_row['recommended_ticker']
    entry_date = pd.to_datetime(entry_row['date'])
    entry_price = entry_row['close_price']

    exit_date = pd.to_datetime(exit_row['date'])
    exit_price = exit_row['close_price']

    if not pd.isna(entry_price) and not pd.isna(exit_price) and entry_price > 0:
        num_shares = np.floor(investment_amount / entry_price)
        profit = (exit_price - entry_price) * num_shares
        trades_list.append({
            'ticker': entry_ticker,
            'entry_date': entry_date,
            'entry_price': entry_price,
            'exit_date': exit_date,
            'exit_price': exit_price,
            'num_shares': num_shares,
            'profit': profit,
            'holding_days': (exit_date - entry_date).days + 1
        })

df_trades = pd.DataFrame(trades_list)

print("\nCalculated Trades and Profits:")
display(df_trades.head(10))
print(f"\nTotal Number of Trades: {len(df_trades)}")
print(f"Total Profit/Loss: ${df_trades['profit'].sum():,.2f}")



Calculated Trades and Profits:


,ticker,entry_date,entry_price,exit_date,exit_price,num_shares,profit,holding_days
0,GLD,2020-01-27,148.990005,2020-01-29,148.460007,33.0,-17.489960,3
1,TLT,2020-01-30,120.031151,2020-01-30,120.031151,41.0,0.000000,1
2,GLD,2020-01-31,149.330002,2020-01-31,149.330002,33.0,0.000000,1
3,TLT,2020-02-03,121.028900,2020-02-03,121.028900,41.0,0.000000,1
4,QQQ,2020-02-04,219.217209,2020-02-11,223.592499,22.0,96.256378,8
5,DIA,2020-02-12,263.811371,2020-02-13,263.008484,18.0,-14.451965,2
6,QQQ,2020-02-14,226.127045,2020-02-18,226.213791,22.0,1.908417,5
7,GLD,2020-02-19,151.789993,2020-02-24,156.089996,32.0,137.600098,6
8,TLT,2020-02-25,125.610924,2020-02-25,125.610924,39.0,0.000000,1
9,GLD,2020-02-26,153.970001,2020-02-26,153.970001,32.0,0.000000,1



Total Number of Trades: 587
Total Profit/Loss: $13,335.68


In [29]:
investment_amount = 5000 # As specified by the user
yearly_profit_summary['percent_return'] = (yearly_profit_summary['profit'] / investment_amount) * 100

print("\nYearly Profit Summary with Percentage Return:")
display(yearly_profit_summary.round(2))


Yearly Profit Summary with Percentage Return:


,year,profit,percent_return
0,2020,3655.57,73.11
1,2021,1844.12,36.88
2,2022,1004.37,20.09
3,2023,1967.72,39.35
4,2024,2167.60,43.35
5,2025,2696.30,53.93
